# IUCN Ecosystem Red List Assessment - Data Processing

Processes raw RLE assessment data to produce:
1. A GeoJSON layer for Mapbox upload (replacing `globalmangrovewatch.20gft2fx`, source-layer `ecoregions_data`)
2. `ecoregions.csv` for backoffice widget table
3. `ecoregion_reports.csv` for backoffice reports table

### Input files (in `../../data/raw/`):
- `RLE_20260831.xlsx` — Assessment criteria per province (36 provinces)
- `Mangrove_RLE.zip` — Province boundaries (shapefile)
- `Individual.assessments.and.associated.DOIs.docx` — Report links with DOIs

### Field mapping notes
The popup component (`map-popup.tsx`) maps field suffixes `_1`, `_2`, `_3` to tags in this order:
- Reduction / Env degradation / Biotic processes: `_1` = Historical (1750), `_2` = Past 50 years, `_3` = Future
- Restricted geographic distribution: `_1` = Extent of occurrence, `_2` = Area of occupancy, `_3` = Threat locations
- Quantitative risk analysis: `_1` = single value

The Excel columns are in a different order (Past, Future, Historical), so we reorder during mapping.

In [2]:
import pandas as pd
import geopandas as gpd
from docx import Document
from pathlib import Path

RAW_DIR = Path('../../data/raw')
OUT_DIR = Path('../../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Excel has a two-row header (merged group headers + sub-headers).
# Skip both and assign descriptive column names.
df = pd.read_excel(RAW_DIR / 'RLE_20260831.xlsx', header=None, skiprows=2)

df.columns = [
    'province',
    # Reduction in Geographic Distribution
    'red_past50', 'red_future', 'red_hist',
    # Restricted Geographic Distribution
    'res_extent', 'res_area', 'res_threat',
    # Environmental Degradation
    'env_past50', 'env_future', 'env_hist',
    # Distribution of Biotic Processes
    'bio_past50', 'bio_future', 'bio_hist',
    # Quantitative Risk Analysis & Overall
    'qra', 'overall',
]

# Normalize province names to match shapefile
df['province'] = df['province'].replace({
    'Sahelian': 'The Sahelian',
    'Arabian (Persian) Gulf': 'Arabian-Persian Gulf',
})

print(f'Loaded {len(df)} provinces from Excel')
df.head()

In [ ]:
gdf = gpd.read_file(f'zip://{RAW_DIR / "Mangrove_RLE.zip"}')
print(f'Loaded {len(gdf)} features from shapefile')
print(f'CRS: {gdf.crs}')
gdf[['unit_name', 'shortname', 'Region', 'Overall']].head(10)

In [ ]:
doc = Document(RAW_DIR / 'Individual.assessments.and.associated.DOIs.docx')

reports = []
for para in doc.paragraphs:
    text = para.text.strip()
    if 'doi.org' in text:
        parts = text.rsplit('https://', 1)
        name = parts[0].strip()
        url = 'https://' + parts[1].strip()
        reports.append({'name': f'RLE {name} (pdf)', 'url': url})

reports_df = pd.DataFrame(reports)

# Normalize report names to match shapefile unit_name values.
# The popup matches by: unit_name.replace("Mangroves of","").trim()
# then report.name.includes(trimmed), so report names must contain
# the exact substring from unit_name.
REPORT_NAME_FIXES = {
    'Galápagos': 'Galapagos',
    'Gulf of Guinea – North': 'Gulf of Guinea - North',
    'Gulf of Guinea – South': 'Gulf of Guinea - South',
    'the Arabian (Persian) Gulf': 'Arabian (Persian) Gulf',
    'Northwest Australia and Sahul Shelf': 'North-west Australia and Sahul shelf',
    'Southwest Australian Shelf': 'South-west Australian shelf',
    'Tropical Northwest Atlantic': 'Tropical Northwestern Atlantic',
    'Tropical Northwest Pacific': 'Tropical Northwestern Pacific',
    'Tropical Southwest Atlantic': 'Tropical Southwestern Atlantic',
    'Tropical Southwest Pacific': 'Tropical Southwestern Pacific',
    'Warm Temperate Southwest Atlantic': 'Warm Temperate Southwestern Atlantic',
    'South India, Sri Lanka, and Maldives': 'South India, Sri Lanka and Maldives',
}

for old, new in REPORT_NAME_FIXES.items():
    reports_df['name'] = reports_df['name'].str.replace(old, new, regex=False)

print(f'Found {len(reports_df)} reports')
reports_df

In [ ]:
rename_map = {
    # Reduction in Geographic Distribution
    'red_hist':    'reduction_in_geographic_distribution_1',   # Historical (1750)
    'red_past50':  'reduction_in_geographic_distribution_2',   # Past 50 years
    'red_future':  'reduction_in_geographic_distribution_3',   # Future
    # Restricted Geographic Distribution
    'res_extent':  'restricted_geographic_distribution_1',     # Extent of occurrence
    'res_area':    'restricted_geographic_distribution_2',     # Area of occupancy
    'res_threat':  'restricted_geographic_distribution_3',     # Threat locations <5
    # Environmental Degradation
    'env_hist':    'environmental_degradation_1',              # Historical (1750)
    'env_past50':  'environmental_degradation_2',              # Past 50 years
    'env_future':  'environmental_degradation_3',              # Future
    # Distribution of Biotic Processes
    'bio_hist':    'distribution_of_biotic_processes_1',       # Historical (1750)
    'bio_past50':  'distribution_of_biotic_processes_2',       # Past 50 years
    'bio_future':  'distribution_of_biotic_processes_3',       # Future
    # Quantitative Risk Analysis
    'qra':         'quantitative_risk_analysis_1',
    # Overall
    'overall':     'overall_assessment',
}

assessments = df.rename(columns=rename_map)
assessments.head()

In [ ]:
# Verify province names match, then join
excel_provinces = set(assessments['province'])
shp_provinces = set(gdf['shortname'])
assert excel_provinces == shp_provinces, f'Name mismatch: {excel_provinces ^ shp_provinces}'
print(f'All {len(excel_provinces)} province names match')

layer = gdf.merge(assessments, left_on='shortname', right_on='province', how='left')

layer_fields = [
    'Region', 'unit_name', 'overall_assessment',
    'reduction_in_geographic_distribution_1',
    'reduction_in_geographic_distribution_2',
    'reduction_in_geographic_distribution_3',
    'restricted_geographic_distribution_1',
    'restricted_geographic_distribution_2',
    'restricted_geographic_distribution_3',
    'environmental_degradation_1',
    'environmental_degradation_2',
    'environmental_degradation_3',
    'distribution_of_biotic_processes_1',
    'distribution_of_biotic_processes_2',
    'distribution_of_biotic_processes_3',
    'quantitative_risk_analysis_1',
    'geometry',
]

layer_out = layer[layer_fields].copy()
print(f'Layer features: {len(layer_out)}')
layer_out.drop(columns='geometry').head()

In [ ]:
# Backoffice CSVs
overall_counts = assessments['overall_assessment'].str.lower().value_counts()
categories = ['dd', 'lc', 'nt', 'vu', 'en', 'cr', 'ne']

ecoregions = pd.DataFrame({
    'indicator': 'ecoregion_asssessment',  # NOTE: triple-s typo matches existing data
    'category': categories,
    'value': [float(overall_counts.get(c, 0)) for c in categories],
})
ecoregions.to_csv(OUT_DIR / 'ecoregions.csv', index=False)

reports_df.to_csv(OUT_DIR / 'ecoregion_reports.csv', index=False)

print(f'ecoregions.csv: {len(categories)} categories, {ecoregions["value"].sum():.0f} total')
print(f'ecoregion_reports.csv: {len(reports_df)} reports')
ecoregions

In [ ]:
import subprocess

# Reproject to WGS84 (tippecanoe requires EPSG:4326) and export GeoJSON
layer_4326 = layer_out.to_crs(epsg=4326)
output_path = OUT_DIR / 'iucn_ecoregion_layer.geojson'
layer_4326.to_file(output_path, driver='GeoJSON')

mbtiles_path = OUT_DIR / 'iucn_ecoregion_layer.mbtiles'
subprocess.run(
    f"tippecanoe -zg -f -P -l ecoregions_data -o {mbtiles_path} --extend-zooms-if-still-dropping {output_path}",
    shell=True,
    check=True,
)

print(f'GeoJSON: {output_path} ({len(layer_4326)} features)')
print(f'MBTiles: {mbtiles_path}')

In [ ]:
# Validation
assert layer_out['overall_assessment'].isna().sum() == 0, 'Null overall_assessment found'
print('Overall assessment distribution:')
print(layer_out['overall_assessment'].value_counts().sort_index())

# Check report matching (popup strips "Mangroves of" and uses includes())
unmatched = []
for _, row in layer_out.iterrows():
    uname = row['unit_name']
    if pd.isna(uname):
        continue
    trimmed = uname.replace('Mangroves of', '').strip()
    if not reports_df['name'].str.contains(trimmed, regex=False).any():
        unmatched.append((uname, row['overall_assessment']))

if unmatched:
    print(f'\nNo matching report for {len(unmatched)} provinces:')
    for name, oa in unmatched:
        print(f'  {name} (overall: {oa})')
else:
    print('\nAll provinces have matching reports')

## Frontend changes required

The data now uses standard IUCN abbreviations. The frontend needs these updates:

1. **CR instead of CE**: Replace all `ce`/`CE` references with `cr`/`CR` in:
   - `types.d.ts` — `CategoryIds` type
   - `constants.ts` — `COLORS` and `LABELS` objects
   - `hooks.tsx` — `OVERALL_ASSESSMENT` paint map

2. **Add NE (Not Evaluated)** and **NT (Near Threatened)** support:
   - Add to `CategoryIds` type
   - Add to `COLORS`: `nt: '#F3AD6A'`, `ne: '#ECECEF'` (NE color already exists)
   - Add to `LABELS`: `nt: 'Near Threatened'`, `ne: 'Not Evaluated'`
   - Add to `OVERALL_ASSESSMENT` paint map in `hooks.tsx`:
     ```js
     const OVERALL_ASSESSMENT = {
       CR: '#EE4D5A',  // was CE
       EN: '#F97B57',
       VU: '#ECDA9A',
       NT: '#F3AD6A',  // new
       LC: '#B4DCAA',
       DD: '#ECECEF',
       NE: '#ECECEF',  // new
     };
     ```
   - Remove the NT filter in `hooks.tsx`: `data?.filter((d) => d.category !== 'nt')`
   - Update popup legend items in `map-popup.tsx` to include all categories